CNN1D



In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import random
from sklearn.model_selection import train_test_split
import hashlib
import copy
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
import subprocess
import shutil
from tqdm import tqdm

I0000 00:00:1786002996.163450  911914 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [3]:
SEEDS = [42, 8, 1291, 64207, 305, 91876, 12, 456, 33920, 7]

In [4]:
MODEL_NAME="CNN1D"
N_MELS=128
TIME_STEPS=32  #157 se 5s

DATASET_CSV_FILE = f"audio_dataset.csv"
SPECTROGRAMS_DIR = f"Spectrograms"
MAIN_METRIC="mae" #mse
START_FROM_INDEX=0  #0
START_FROM_SEED_INDEX=0 #0

In [5]:
def create_model(input_shape=(32, 128), dropout_rate=0.2, kernel_size=3):
    model = models.Sequential(name=MODEL_NAME)
    model.add(layers.Input(shape=input_shape))

    # --- Blocco 1: Estrattore locale (basso livello) ---
    model.add(layers.Conv1D(filters=32, kernel_size=kernel_size, padding='same', activation=None))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU(max_value=6.0))
    model.add(layers.MaxPooling1D(pool_size=2))

    # --- Blocco 2: Estrattore intermedio ---
    model.add(layers.Conv1D(filters=64, kernel_size=kernel_size, padding='same', activation=None))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU(max_value=6.0))
    model.add(layers.MaxPooling1D(pool_size=2))

    # --- Blocco 3: Estrattore profondo (sostituisce la memoria della LSTM) ---
    # Usiamo una convoluzione con kernel più largo o dilated per "vedere" più lontano nel tempo
    model.add(layers.Conv1D(filters=128, kernel_size=kernel_size, padding='same', activation=None))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU(max_value=6.0))

    # --- Sostituzione LSTM: Global Average Pooling ---
    # Invece di una LSTM, usiamo il Pooling per sintetizzare l'intero segnale
    model.add(layers.GlobalAveragePooling1D())

    # --- Regressore Finale ---
    model.add(layers.Dense(64, activation=None))
    model.add(layers.ReLU(max_value=6.0))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation=None))

    return model

In [6]:
def create_model_for_deploy(trained_model, input_shape=(32, 128), dropout_rate=0.2, kernel_size=3):
    """
    Input_shape: (passi_temporali, caratteristiche_frequenza)
    """
    model = models.Sequential(name=MODEL_NAME)

    # 1. Porta d'ingresso esplicita
    model.add(layers.Input(shape=input_shape, batch_size=1))

    # --- Blocco 1: Estrattore locale (basso livello) ---
    model.add(layers.Conv1D(filters=32, kernel_size=kernel_size, padding='same', activation=None))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU(max_value=6.0))
    model.add(layers.MaxPooling1D(pool_size=2))

    # --- Blocco 2: Estrattore intermedio ---
    model.add(layers.Conv1D(filters=64, kernel_size=kernel_size, padding='same', activation=None))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU(max_value=6.0))
    model.add(layers.MaxPooling1D(pool_size=2))

    # --- Blocco 3: Estrattore profondo (sostituisce la memoria della LSTM) ---
    # Usiamo una convoluzione con kernel più largo o dilated per "vedere" più lontano nel tempo
    model.add(layers.Conv1D(filters=128, kernel_size=kernel_size, padding='same', activation=None))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU(max_value=6.0))

    # --- Sostituzione LSTM: Global Average Pooling ---
    # Invece di una LSTM, usiamo il Pooling per sintetizzare l'intero segnale
    model.add(layers.GlobalAveragePooling1D())

    # --- Regressore Finale ---
    model.add(layers.Dense(64, activation=None))
    model.add(layers.ReLU(max_value=6.0))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation=None))

    model.set_weights(trained_model.get_weights())

    return model

In [7]:
class SpectrogramDataset:
    def __init__(self, csv_file, bin_dir, x_mean, x_std, y_mean, y_std):
        """
        Pipeline di caricamento Spettrogrammi ottimizzata per TensorFlow.
        Modificata per file BINARI (.bin) e architetture CNN + LSTM.
        """
        self.annotations = pd.read_csv(csv_file)
        self.bin_dir = bin_dir

        self.x_mean, self.x_std = x_mean, x_std
        self.y_mean, self.y_std = y_mean, y_std

        # Poiché i file .bin non hanno un header con le dimensioni,
        # dobbiamo passarle esplicitamente o definirle come costanti.
        self.altezza = N_MELS       # N_MELS (Caratteristiche di frequenza)
        self.larghezza = TIME_STEPS  # TIME_STEPS (Istantanee temporali)


        self.input_shape_modello = (self.larghezza, self.altezza)

        print(f"Proprietà Dataset -> Dimensioni native spettrogramma (H x W): {self.altezza}x{self.larghezza}")
        print(f"Risoluzione di input impostata a: {self.input_shape_modello} (Tempo x Frequenze)")

        # Prepariamo le liste complete dei percorsi dei file .bin e dei target (labels)
        self.file_paths = [
            os.path.join(self.bin_dir, f"{os.path.splitext(f)[0]}_melspec.bin")
            for f in self.annotations['filename'].values
        ]
        self.labels = self.annotations['portata'].values.astype(np.float32)

    def _parse_function(self, filename, label):
        """Funzione interna che TensorFlow userà per caricare e ricostruire ogni singolo file binario"""
        # 1. Legge il file in formato binario raw (stringa di byte)
        binary_string = tf.io.read_file(filename)

        # 2. Decodifica i byte grezzi direttamente in un vettore di float32
        # (visto che in Python li abbiamo salvati come np.float32)
        flat_tensor = tf.io.decode_raw(binary_string, out_type=tf.float32)

        # 3. Ricostruiamo la matrice (reshape) usando lo shape nativo di Librosa: [N_MELS, TIME_STEPS]
        matrix = tf.reshape(flat_tensor, [self.altezza, self.larghezza])

        # 2. Sostituisci il Min-Max con lo Z-Score (Standardizzazione)
        # Standardizzazione Input (Spettrogramma)
        matrix = (matrix - self.x_mean) / (self.x_std + 1e-6)

        # Standardizzazione Label (Target)
        label = (label - self.y_mean) / (self.y_std + 1e-6)

        # Trasponiamo la matrice (Invertiamo Righe e Colonne)
        # Da [Altezza, Larghezza] diventa [Larghezza, Altezza] -> [Tempo, Frequenze]
        matrix = tf.transpose(matrix, perm=[1, 0])

        # Assicuriamo esplicitamente lo shape statico richiesto dal modello
        matrix.set_shape([self.larghezza, self.altezza])

        return matrix, label

    def get_dataset(self, batch_size=32, shuffle=True, buffer_size=1000):
        """Genera e restituisce l'oggetto tf.data.Dataset pronto per model.fit()"""
        dataset = tf.data.Dataset.from_tensor_slices((self.file_paths, self.labels))

        if shuffle:
            dataset = dataset.shuffle(buffer_size=buffer_size)

        # .map() applica il caricamento binario e la trasposizione in parallelo
        dataset = dataset.map(self._parse_function, num_parallel_calls=tf.data.AUTOTUNE)

        # Raggruppa in batch. I batch avranno shape: (Batch_Size, Tempo, Frequenze)
        dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

        return dataset

In [8]:
def train_model(model, epochs, train_dataset, val_dataset, learning_rate, patience=3):
    """
    Addestra il modello TensorFlow usando l'API Keras e i Callbacks per l'Early Stopping.

    Parametri:
    - model: Il modello TensorFlow Keras compilato.
    - epochs: Numero massimo di epoche.
    - train_dataset: Oggetto tf.data.Dataset di training (restituito da get_dataset()).
    - val_dataset: Oggetto tf.data.Dataset di validazione.
    - optimizer: Ottimizzatore Keras (es. tf.keras.optimizers.Adam).
    - criterion: Loss function (es. 'mse' o 'mae' o un'istanza di tf.keras.losses.Loss).
    - patience: Numero di epoche da attendere prima di interrompere l'addestramento.
    """
    print("Inizio dell'addestramento...")

    # Configurazione di Ottimizzatore e Loss
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if MAIN_METRIC=="mse":
      criterion = tf.keras.losses.MeanSquaredError()
    else:
      criterion = tf.keras.losses.MeanAbsoluteError()

    # 1. Compiliamo il modello inserendo l'ottimizzatore e la loss configurati
    model.compile(optimizer=optimizer, loss=criterion, metrics=["mae"])

    # 2. Definiamo i Callbacks per emulare esattamente la tua logica PyTorch
    # EarlyStopping interrompe il training e ripristina automaticamente i pesi migliori (restore_best_weights)
    callbacks_list = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience,
            mode='min',
            verbose=1,
            restore_best_weights=True  # Carica automaticamente i pesi dell'epoca migliore alla fine
        )
    ]

    # 3. Avviamo il training (Fasi di Train, Validation ed Early Stopping avvengono qui dentro)
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks_list,
        verbose=1 # Mostra la barra di progresso e le loss per epoca
    )

    # --- 4. ESTRAZIONE DELLE METRICHE DELLO STATO MIGLIORE ---
    # Troviamo l'indice dell'epoca con la minore validation loss
    val_loss_history = history.history['val_loss']
    best_epoch_idx = val_loss_history.index(min(val_loss_history))

    best_epoch = best_epoch_idx + 1
    best_train_loss = history.history['loss'][best_epoch_idx]

    print(f"\nAddestramento terminato. Il modello migliore è stato trovato all'epoca {best_epoch}.")

    return best_train_loss, best_epoch

In [9]:
def evaluate_model(model, dataset, y_mean, y_std):
    """
    Valuta il modello TensorFlow sul test set calcolando MAE, MAPE, RMSE e R²
    ripristinando la scala reale (fisica) tramite Z-Score denormalization.
    """
    print("Valutazione del modello in corso...")

    # 1. Estraiamo tutti i target (labels standardizzate) dal tf.data.Dataset
    y_true_list = []
    for _, labels in dataset:
        y_true_list.append(labels.numpy())
    y_true = np.concatenate(y_true_list, axis=0)

    # 2. Generiamo le predizioni del modello (anche queste sono standardizzate)
    y_pred = model.predict(dataset, verbose=0)

    # Assicuriamoci che entrambi i vettori abbiano lo stesso shape piatto (1D)
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    # --- DENORMALIZZAZIONE: RIPRISTINO DELLA SCALA REALE FISICA (Z-SCORE) ---
    # Formula inversa: x = (z * std) + mean
    y_true_real = (y_true * y_std) + y_mean
    y_pred_real = (y_pred * y_std) + y_mean

    # 3. Calcolo delle metriche di regressione REALI tramite scikit-learn
    mae = mean_absolute_error(y_true_real, y_pred_real)
    mape = mean_absolute_percentage_error(y_true_real, y_pred_real) * 100
    mse = mean_squared_error(y_true_real, y_pred_real)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_real, y_pred_real)

    metrics = {
        "mae": float(mae),
        "mape": float(mape),
        "mse": float(mse),
        "rmse": float(rmse),
        "r2": float(r2)
    }

    return metrics

In [10]:
# --- 4. FUNZIONE INTERNA PER ESEGUIRE XXD E GENERARE SOLO IL .CPP ---
def execute_xxd(tflite_file_path, base_name, models_dir):
    # Verifica preventiva se xxd è effettivamente installato nel sistema operativo sottostante
    if not shutil.which("xxd"):
        print(f"\n[ATTENZIONE] Comando 'xxd' non trovato nel sistema!")
        print("Per installarlo esegui in una cella libera:")
        print("!sudo apt-get update && sudo apt-get install -y xxd  <- (Se sei su Linux/Colab)")
        print("!brew install xxd                                 <- (Se sei su macOS)")
        return

    # Puliamo il nome della variabile C++ rimuovendo caratteri non validi
    var_name = base_name.replace("-", "_").replace(".", "_")

    # Se vuoi generare SOLO il file .h al posto del .cpp, ti basta decommentare h_out
    # e commentare cpp_out (cambiando poi la logica di scrittura sotto).
    cpp_filename=f"{base_name}.cpp"
    cpp_out = models_dir / cpp_filename

    file_size=0

    try:
        # Comando xxd standard: -i genera l'array in stile C, -name cambia la variabile
        comando = f'xxd -i {tflite_file_path}'

        # Eseguiamo il comando e catturiamo il testo (specificando UTF-8 per evitare bug di Windows)
        risultato = subprocess.run(comando, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding='utf-8')
        output_grezzo = risultato.stdout

        # CORREZIONE RAM ESP32: Sostituiamo le definizioni aggiungendo 'const'
        output_ottimizzato = output_grezzo.replace("unsigned char", "const unsigned char")
        output_ottimizzato = output_ottimizzato.replace("unsigned int", "const unsigned int")

        # Scrittura del file .cpp definitivo (Senza includere l'header .h in cima)
        with open(cpp_out, "w", encoding="utf-8") as cpp_file:
            cpp_file.write("#include <stddef.h>\n\n")  # Include standard per l'array
            cpp_file.write(output_ottimizzato)

        print(f"   -> Generato file C++: {cpp_out.name}")

        file_size = cpp_out.stat().st_size

    except subprocess.CalledProcessError as err:
        print(f"[ERRORE] Errore durante l'esecuzione di xxd: {err.stderr}")

    return cpp_filename,file_size

In [11]:
def save_results(data, report_csv_path):
  # 3. Convertiamo in DataFrame di Pandas
  df_risultato = pd.DataFrame(data)

  # 4. Salviamo in CSV usando i metodi nativi di Path
  # .exists() controlla se il file c'è già per decidere se fare l'append ('a') o la scrittura ('w')
  if report_csv_path.exists():
      df_risultato.to_csv(report_csv_path, mode='a', header=False, index=False)
  else:
      # Se il file non esiste, ci assicuriamo prima che la cartella di destinazione esista davvero
      df_risultato.to_csv(report_csv_path, mode='w', header=True, index=False)


In [12]:
def save_model(model, index, train_dataset, models_dir):
    """
    Converte un modello CNN classico in due formati TensorFlow Lite distinti:
    1. Versione Standard (Float32) - Pura senza Flex Ops
    2. Versione Quantizzata (Int8) - Full Integer nativa al 100% per ESP32.

    Restituisce una tupla con i nomi dei file e le rispettive dimensioni in byte.
    """

    model.save(models_dir / f"{MODEL_NAME}_{index}.keras")

    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    tf.get_logger().setLevel('FATAL')

    # --- 1. SETUP DEI PERCORSI ---
    f32_filename = f"{MODEL_NAME}_{index}_float32.tflite"
    f32_path = models_dir / f32_filename

    int8_filename = f"{MODEL_NAME}_{index}_int8.tflite"
    int8_path = models_dir / int8_filename

    # --- 2. PRIMA CONVERSIONE: VERSIONE STANDARD (FLOAT32) ---
    print(f"Esportazione versione FLOAT32 per esperimento {index}...")

    converter_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
    # Per la CNN usiamo SOLO i builtins standard (niente SELECT_TF_OPS)
    converter_f32.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

    tflite_f32_model = converter_f32.convert()

    with open(f32_path, "wb") as f:
        f.write(tflite_f32_model)
    #f32_size = f32_path.stat().st_size


    # --- 3. SECONDA CONVERSIONE: VERSIONE QUANTIZZATA (INT8 STRITTA) ---
    print(f"Esportazione versione QUANTIZZATA INT8 per esperimento {index}...")

    converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
    converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

    # Generatore di calibrazione (100 campioni reali sono perfetti per la CNN)
    def representative_data_gen():
      # Estraiamo esplicitamente solo la componente x (input_value) ignorando la y (_)
      for input_value, _ in train_dataset.take(10):
        for i in range(input_value.shape[0]):
          sample = tf.expand_dims(input_value[i], axis=0)
          # Garantisce che i dati siano float32 prima della quantizzazione INT8
          yield [tf.cast(sample, tf.float32)]

    converter_int8.representative_dataset = representative_data_gen

    # FORZATURA FULL INTEGER: Rifiuta la conversione se qualche livello non è quantizzabile in INT8 hardware
    converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

    # Blocca l'I/O a 8 bit (obbligatorio per ESP32)
    converter_int8.inference_input_type = tf.int8
    converter_int8.inference_output_type = tf.int8

    tflite_int8_model = converter_int8.convert()

    with open(int8_path, "wb") as f:
        f.write(tflite_int8_model)
    #int8_size = int8_path.stat().st_size

    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'

    f32_filename,f32_size=execute_xxd(f32_path, f"{MODEL_NAME}_{index}_float32", models_dir=models_dir)
    int8_filename,int8_size=execute_xxd(int8_path, f"{MODEL_NAME}_{index}_int8", models_dir=models_dir)

    # --- 4. RITORNO DEI RISULTATI ---
    return f32_filename, f32_size, int8_filename, int8_size

In [13]:
def set_seed(seed=42):
    """Fissa il seme ovunque in TensorFlow per garantire la riproducibilità"""
    # 1. Imposta il seed per Python, NumPy e TensorFlow in un colpo solo
    tf.keras.utils.set_random_seed(seed)

    # 2. Forza TensorFlow ad usare operazioni deterministiche sulla GPU
    # Questo equivale a `torch.backends.cudnn.deterministic = True`
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

    # Opzionale: disattiva il parallelismo interno se vuoi un determinismo microscopico
    # (Nota: questo potrebbe rallentare leggermente l'addestramento su GPU)
    # tf.config.experimental.enable_op_determinism()

In [14]:
def calculate_stats(file_paths, labels, batch_size=64):
    # Creiamo un dataset tf.data per leggere i file in parallelo
    def process_path(file_path):
        raw = tf.io.read_file(file_path)
        return tf.io.decode_raw(raw, out_type=tf.float32)

    dataset = tf.data.Dataset.from_tensor_slices(file_paths)
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

    sum_x = 0.0
    sum_sq_x = 0.0
    count_x = 0

    # Calcoliamo il numero totale di batch per la barra
    total_batches = (len(file_paths) + batch_size - 1) // batch_size

    # Wrapping del loop con tqdm per la barra di progresso
    print("Calcolo statistiche in corso...")
    for batch in tqdm(dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE), total=total_batches):
        batch_val = batch.numpy()
        sum_x += np.sum(batch_val)
        sum_sq_x += np.sum(np.square(batch_val))
        count_x += batch_val.size

    mean_x = sum_x / count_x
    std_x = np.sqrt(max(sum_sq_x / count_x - mean_x**2, 1e-9))

    mean_y = np.mean(labels)
    std_y = np.std(labels)

    return mean_x, std_x, mean_y, std_y

In [ ]:
for i, seed in enumerate(SEEDS[START_FROM_SEED_INDEX:], start=START_FROM_SEED_INDEX):

    if START_FROM_SEED_INDEX != 0:
        print(f"Skipping the first {START_FROM_SEED_INDEX} seeds...")

    print(f"\n\n=== INIZIO RANDOMIZED SEARCH CON SEME {seed} ({i+1}/{len(SEEDS)}) ===\n")

    REPORT_CSV = Path(f"{MODEL_NAME}ResultsForMicro_{seed}.csv")
    MODELS_DIR=Path(f"{MODEL_NAME}ToEvaluate_{seed}")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    set_seed(seed)
    # --- 3. FIXED INDEX SPLIT (MATCHING ML MODELS) ---
    # Read the CSV to get the exact same row structure
    temp_df = pd.read_csv(DATASET_CSV_FILE)

    # Generiamo due liste di indici (0, 1, 2...) basandoci sulla struttura del DataFrame
    indices = list(range(len(temp_df)))

    # Usiamo lo STESSO IDENTICO train_test_split del Machine Learning sul vettore degli indici
    train_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=seed)

    print("SHA256 Train1: "+hashlib.sha256(str(train_indices).encode('utf-8')).hexdigest())
    print("SHA256 Test1: "+hashlib.sha256(str(test_indices).encode('utf-8')).hexdigest())

    train_indices, val_indices = train_test_split(
        train_indices,
        test_size=0.15,          # 15% dei dati di train va alla validazione
        random_state=seed,
        shuffle=True             # Mescoliamo gli indici prima di dividerli
    )


    # Converto in liste per iterare velocemente
    train_filenames = temp_df.iloc[train_indices]['filename'].values
    train_labels = temp_df.iloc[train_indices]['portata'].values

    # Uso:
    file_paths = [os.path.join(SPECTROGRAMS_DIR, f.replace('.wav', '_melspec.bin')) for f in train_filenames]
    mean_x, std_x, mean_y, std_y = calculate_stats(file_paths, train_labels)

    print(f"Stats Input -> Mean: {mean_x}, Std: {std_x}")
    print(f"Stats Labels -> Mean: {mean_y}, Std: {std_y}")

    # --- 1. DEFINIAMO LO SPAZIO DI RICERCA ---
    NUM_HYPERPARAMETER_COMBINATIONS = 50 #20
    EPOCHS_PER_TEST = 50  #20
    PATIENCE=8  #5

    #scale_factor_options = [1.0]  # scale_factor_options = [0.05, 0.1, 0.15, 0.2]
    batch_size_options = [16, 32, 64]
    lr_interval = [5e-5, 1e-4, 3e-4, 5e-4]
    dropout_options = [0.1, 0.2, 0.3, 0.4]
    kernel_options = [3, 5]

    # --- 2. GENERAZIONE DELLE N COMBINAZIONI UNICHE ---
    random.seed(42)     #LASCIA 42! (combinazioni uguali per tutti i seed)
    hyperparameter_combinations = []
    while len(hyperparameter_combinations) < NUM_HYPERPARAMETER_COMBINATIONS:
        combo = {
            #"Scale_Factor": random.choice(scale_factor_options),
            "Batch_Size": random.choice(batch_size_options),
            "Learning_Rate": random.choice(lr_interval),
            "Dropout": random.choice(dropout_options),
            "Kernel_Size": random.choice(kernel_options),
        }
        if combo not in hyperparameter_combinations:
            hyperparameter_combinations.append(combo)


    # --- PLANNED COMBINATIONS ---
    print("=" * 50)
    print(f"GENERATED {NUM_HYPERPARAMETER_COMBINATIONS} UNIQUE HYPERPARAMETER COMBINATIONS:")
    print("=" * 50)
    for idx, combo in enumerate(hyperparameter_combinations):
        print(f"  Configuration {idx + 1}:")
        #print(f"    • Scale Factor:  {combo['Scale_Factor']}")
        print(f"    • Batch Size:    {combo['Batch_Size']}")
        print(f"    • Learning Rate: {combo['Learning_Rate']}")
        print(f"    • Dropout:       {combo['Dropout']}")
        print(f"    • Kernel size:   {combo['Kernel_Size']}")
        print("-" * 30)
    print("=" * 50 + "\n")

    if START_FROM_INDEX != 0:
        print(f"Skipping the first {START_FROM_INDEX} combinations...")
    
    # --- 2. IL CICLO DELLA RANDOMIZED SEARCH (VERSIONE TENSORFLOW) ---
    for i, combo in enumerate(hyperparameter_combinations[START_FROM_INDEX:], start=START_FROM_INDEX):

        # Estraiamo i parametri per questo esperimento
        #current_scale = combo["Scale_Factor"]
        current_batch = combo["Batch_Size"]
        current_lr = combo["Learning_Rate"]
        current_drop = combo["Dropout"]
        current_kernel_size = combo["Kernel_Size"]

        print(f"\n=== AVVIO ESPERIMENTO {i+1}/{NUM_HYPERPARAMETER_COMBINATIONS} ===")
        # Sistemata la stringa multilinea per evitare SyntaxError causati da spazi dopo il backslash
        print(f"Parametri scelti -> Batch: {current_batch}, LR: {current_lr}, Dropout: {current_drop}, Kernel size: {current_kernel_size}")

        try:
            # 1. Inizializziamo il gestore del Dataset
            dataset_wrapper = SpectrogramDataset(
                csv_file=DATASET_CSV_FILE,
                bin_dir=SPECTROGRAMS_DIR,
                x_mean = mean_x,
                x_std = std_x,
                y_mean = mean_y,
                y_std = std_y
            )

            # 3. Filtriamo i dati (usiamo i valori RAW, la normalizzazione avviene nella _parse_function)
            train_paths = [dataset_wrapper.file_paths[idx] for idx in train_indices]
            train_labels_raw = [dataset_wrapper.labels[idx] for idx in train_indices]

            val_paths = [dataset_wrapper.file_paths[idx] for idx in val_indices]
            val_labels_raw = [dataset_wrapper.labels[idx] for idx in val_indices]

            test_paths = [dataset_wrapper.file_paths[idx] for idx in test_indices]
            test_labels_raw = [dataset_wrapper.labels[idx] for idx in test_indices]

            # 4. Configura i tf.data.Dataset (usando le liste RAW)
            train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels_raw)) \
                .shuffle(buffer_size=1000) \
                .map(dataset_wrapper._parse_function, num_parallel_calls=tf.data.AUTOTUNE) \
                .batch(current_batch).prefetch(tf.data.AUTOTUNE)

            train_eval_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels_raw)) \
            .map(dataset_wrapper._parse_function, num_parallel_calls=tf.data.AUTOTUNE) \
            .batch(current_batch).prefetch(tf.data.AUTOTUNE)

            val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels_raw)) \
                .map(dataset_wrapper._parse_function, num_parallel_calls=tf.data.AUTOTUNE) \
                .batch(current_batch).prefetch(tf.data.AUTOTUNE)

            test_dataset = tf.data.Dataset.from_tensor_slices((test_paths, test_labels_raw)) \
                .map(dataset_wrapper._parse_function, num_parallel_calls=tf.data.AUTOTUNE) \
                .batch(current_batch).prefetch(tf.data.AUTOTUNE)

            print(f"Dataset preparato correttamente!")
            print(f"Campioni per il training: {len(train_indices)} | Campioni per la validazione: {len(val_indices)} | Campioni per il test: {len(test_indices)}")

            # 2. Inizializzazione dell'architettura in TensorFlow (Formato dell'input H x W x Canali)
            #input_shape = (dataset_wrapper.altezza, dataset_wrapper.larghezza, 1)
            input_shape = dataset_wrapper.input_shape_modello
            print(f"INPUT SHAPE: {input_shape}")
            model = create_model(
                input_shape=input_shape,
                dropout_rate=current_drop,
                kernel_size=current_kernel_size
            )

            # --- 3. CICLO DI TRAINING BREVE ---
            train_last_epoch_loss, train_last_epoch = train_model(
                model=model,
                epochs=EPOCHS_PER_TEST,
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                learning_rate=current_lr,
                patience=PATIENCE
            )
            print("Addestramento terminato!")

            # --- VALUTAZIONE SUL TRAINING SET ---
            train_metrics = evaluate_model(model, train_eval_dataset, y_mean = mean_y,y_std = std_y)

            # --- 4. VALUTAZIONE SUL TEST SET ---
            test_metrics = evaluate_model(model, test_dataset,y_mean = mean_y,y_std = std_y)


            print("\n" + "=" * 60)
            print(f"RISULTATI FINALI ESPERIMENTO")
            print("=" * 60)
            print(f"  • Trained for:           {train_last_epoch} epochs")
            print(f"  • Train Loss ({MAIN_METRIC.upper()}):      {train_last_epoch_loss:.4f}")
            #print(f"  • Test Loss (MAE):       {test_metrics['test_loss']:.5f}")
            print(f"  • Train MAE:              {train_metrics['mae']:.4f}")
            print(f"  • Train MAPE:             {train_metrics['mape']:.2f}%")
            print(f"  • Train MSE:              {train_metrics['mse']:.4f}")
            print(f"  • Train RMSE:             {train_metrics['rmse']:.4f}")
            print(f"  • Train R² Score:         {train_metrics['r2']:.4f}")
            print(f"  • Test MAE:              {test_metrics['mae']:.4f}")
            print(f"  • Test MAPE:             {test_metrics['mape']:.2f}%")
            print(f"  • Test MSE:              {test_metrics['mse']:.4f}")
            print(f"  • Test RMSE:             {test_metrics['rmse']:.4f}")
            print(f"  • Test R² Score:         {test_metrics['r2']:.4f}")
            print("=" * 60 + "\n")

            # --- 5. SALVATAGGIO DIRETTO IN TENSORFLOW LITE ---
            model_for_deploy=create_model_for_deploy(
                input_shape=input_shape,
                dropout_rate=current_drop,
                kernel_size=current_kernel_size,
                trained_model=model
            )


            f32_name, f32_size, int8_name, int8_size = save_model(model_for_deploy, index=i+1, train_dataset=train_dataset, models_dir=MODELS_DIR)

            num_params=model_for_deploy.count_params()

            # --- 6. SALVATAGGIO DEI RISULTATI NEL CSV ---
            data = {
                "Model": [f"{MODEL_NAME}_{i+1}"],
                "Seed":[seed],
                #"Scale_Factor": [current_scale],
                "Batch_Size": [current_batch],
                "Learning_Rate": [current_lr],
                "Dropout": [current_drop],
                "Kernel_Size": [current_kernel_size],
                "Train_Last_Epoch": [train_last_epoch],
                "Train_Last_Epoch_Loss": [train_last_epoch_loss],
                #"Test_Loss": [test_metrics["test_loss"]],
                "MAE_train": [train_metrics["mae"]],
                "MAPE_train": [train_metrics["mape"]],
                "MSE_train": [train_metrics["mse"]],
                "RMSE_train": [train_metrics["rmse"]],
                "R2_train": [train_metrics["r2"]],
                "MAE_test": [test_metrics["mae"]],
                "MAPE_test": [test_metrics["mape"]],
                "MSE_test": [test_metrics["mse"]],
                "RMSE_test": [test_metrics["rmse"]],
                "R2_test": [test_metrics["r2"]],
                "Numero parametri":[num_params],
                "DimensioneModelloF32":[num_params*4],
                "DimensioneModelloInt8":[num_params],
                "FilenameF32": [f32_name],
                "DimensioneF32": [str(f32_size)],
                "FilenameInt8": [int8_name],
                "DimensioneInt8": [str(int8_size)],
            }

            save_results(data, report_csv_path=REPORT_CSV)

        except tf.errors.ResourceExhaustedError as e:
            print(f"Esperimento {i+1} saltato a causa di un errore di memoria (OOM): {e}")
            tf.keras.backend.clear_session()
            continue
        except Exception as e:
            print(f"Si è verificato un errore: {e}")
            continue



=== INIZIO RANDOMIZED SEARCH CON SEME 42 (1/10) ===

SHA256 Train1: 72d2e222075a8f8f1af5150136c80e70a1597099b54f860bb98ff9aa9b9de413
SHA256 Test1: c942b7fdd659fe737f68c6b07db1ce15dee591ea6dc65f83c9d58ac3038fa68c


I0000 00:00:1786002998.505850  911914 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 26519 MB memory:  -> device: 0, name: NVIDIA RTX 5000 Ada Generation, pci bus id: 0000:55:00.0, compute capability: 8.9


Calcolo statistiche in corso...


100%|██████████| 102/102 [00:00<00:00, 1918.75it/s]


Stats Input -> Mean: -60.63391876220703, Std: 12.75280475616455
Stats Labels -> Mean: 0.38464683634876823, Std: 0.16774556446670189
GENERATED 50 UNIQUE HYPERPARAMETER COMBINATIONS:
  Configuration 1:
    • Batch Size:    64
    • Learning Rate: 5e-05
    • Dropout:       0.1
    • Kernel size:   5
------------------------------
  Configuration 2:
    • Batch Size:    16
    • Learning Rate: 0.0001
    • Dropout:       0.2
    • Kernel size:   3
------------------------------
  Configuration 3:
    • Batch Size:    64
    • Learning Rate: 5e-05
    • Dropout:       0.4
    • Kernel size:   3
------------------------------
  Configuration 4:
    • Batch Size:    16
    • Learning Rate: 5e-05
    • Dropout:       0.2
    • Kernel size:   3
------------------------------
  Configuration 5:
    • Batch Size:    64
    • Learning Rate: 5e-05
    • Dropout:       0.2
    • Kernel size:   5
------------------------------
  Configuration 6:
    • Batch Size:    16
    • Learning Rate: 0.0005
  

I0000 00:00:1786003002.135186  912027 cuda_dnn.cc:461] Loaded cuDNN version 92400


102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.4681 - mae: 0.4681 - val_loss: 0.5366 - val_mae: 0.5366
Epoch 2/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.3337 - mae: 0.3337 - val_loss: 0.3411 - val_mae: 0.3411
Epoch 3/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.2967 - mae: 0.2967 - val_loss: 0.2369 - val_mae: 0.2369
Epoch 4/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.2848 - mae: 0.2848 - val_loss: 0.2150 - val_mae: 0.2150
Epoch 5/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.2635 - mae: 0.2635 - val_loss: 0.2023 - val_mae: 0.2023
Epoch 6/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.2540 - mae: 0.2540 - val_loss: 0.1952 - val_mae: 0.1952
Epoch 7/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.2454 - mae: 0.2454 - val_loss: 0.1886 - val_mae: 0.1886
Epoch 8/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.2404 - mae: 0.2404 - val_loss: 0.1867 - val_mae: 0.1867
Epoch 9/50
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: